# 12 — Verification Against `paper_numbers.yaml`

The final gate. Every claim in `../paper_numbers.yaml` is recomputed from the
committed workbooks and compared against its registered value within its
registered tolerance.

Two statuses are handled differently:

| Status | Treatment |
|---|---|
| `verified` | Recomputed independently and asserted. A mismatch is a **FAIL**. |
| `script` | RNG-dependent (split, GBM, permutation). Compared under the documented seeds. |

**Input:** `../paper_numbers.yaml`, both Indic workbooks, the WMT24 workbook
**Output:** `../results/logs/verification.md`

This notebook is a thin wrapper around `../scripts/verify_paper_numbers.py`, so
that the notebook path and the `make verify` path can never drift apart.

## Step 0 — Configuration

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path("..").resolve()
VERIFIER = REPO_ROOT / "scripts" / "verify_paper_numbers.py"
REGISTRY = REPO_ROOT / "paper_numbers.yaml"
LOG = REPO_ROOT / "results" / "logs" / "verification.md"

print("Repository root:", REPO_ROOT.name)
print("Verifier:", VERIFIER.relative_to(REPO_ROOT))
print("Registry:", REGISTRY.relative_to(REPO_ROOT))

## Step 1 — The Registry

A count of what is being checked, and of what is explicitly not.

In [ ]:
import yaml

registry = yaml.safe_load(REGISTRY.read_text())
claims = registry["claims"]

from collections import Counter
counts = Counter(c["status"] for c in claims)
print(f"  claims registered : {len(claims)}")
for status in ("verified", "script"):
    print(f"    {status:9s}: {counts.get(status, 0)}")
print()
print("  bases:", registry["meta"]["bases"])
print("  seeds:", registry["meta"]["seeds"])
print("  language order:", registry["meta"]["language_order"])

## Step 2 — Run the Verifier

Exits non-zero on any failure. The full per-claim log is written to
`../results/logs/verification.md`.

In [ ]:
proc = subprocess.run([sys.executable, str(VERIFIER)], cwd=REPO_ROOT,
                      capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print("--- stderr ---")
    print(proc.stderr)
print(f"exit code: {proc.returncode}")

## Step 3 — Assert the Gate

In [ ]:
assert proc.returncode == 0, (
    f"verification failed with exit code {proc.returncode}; "
    f"see {LOG.relative_to(REPO_ROOT)}"
)
print("\u2713 Every registered claim reproduced within tolerance")
print(f"\u2713 Log written to {LOG.relative_to(REPO_ROOT)}")

## Step 4 — Output Manifest

In [ ]:
print("=== Notebook 12 — output manifest ===")
print(f"  {LOG.name}  ({LOG.stat().st_size:,} bytes)")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1